# QLoRA fine-tune: Qwen3-4B → crossword-generator SLM

Distills the (Claude + verifier + scaffolding) pipeline into one-shot generation. Trains on `data/sft/train.jsonl` (chat: fixed system contract → minimal size-routed user prompt → verified assistant program), **response-only loss**, dev for validation, `eval` held out for the base-vs-tuned test (see `colab_eval_tuned.ipynb`).

## 1. Install (pinned, Qwen3-capable snapshot)
Versions are **pinned**, not `>=`, on purpose: the current `trl` (1.x) **removed** `DataCollatorForCompletionOnlyLM` and **renamed** `SFTConfig(max_seq_length=)` -> `max_length=`, which would break cells 6-7 with `-U`. `trl==0.19.1` is the last release that supports **both** Qwen3 (needs `transformers>=4.51`) and the response-only collator this notebook relies on.

In [ ]:
!pip install -q -U 'transformers==4.53.*' 'trl==0.19.1' 'peft==0.16.*' 'bitsandbytes==0.46.*' 'accelerate==1.8.*' 'datasets==3.6.*'

> **Expected pip warning -- safe to ignore.** You'll likely see a resolver complaint that Colab's pre-installed `gradio` wants `huggingface-hub>=1.2` but `huggingface-hub 0.36.x` is installed. That 0.3x version is **required** by `transformers 4.53`, and `gradio` is **not used** anywhere in this notebook, so the conflict is cosmetic -- the install still succeeds. **Do not upgrade `huggingface-hub`** (it would break `transformers`).

## 1b. Preflight: confirm the GPU **before** training
Colab Pro only helps if you actually got a bigger card — Pro can still hand you a ~16 GB T4, which will OOM this config (especially the fp16 merge in cell 8). This prints the GPU name + free VRAM and warns if you're under ~20 GB. Want **L4 (24 GB)** or **A100 (40 GB)**: Runtime -> Change runtime type.

In [ ]:
import os
# Set BEFORE torch initializes CUDA: lets the allocator grow segments instead of
# fragmenting (the "reserved but unallocated" memory in OOM tracebacks).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
assert torch.cuda.is_available(), "No GPU attached -- Runtime > Change runtime type > GPU (L4 or A100)."
free_b, total_b = torch.cuda.mem_get_info()
gb = 1024 ** 3
name = torch.cuda.get_device_properties(0).name
total_gb, free_gb = total_b / gb, free_b / gb
print(f"GPU: {name}")
print(f"VRAM: {total_gb:.1f} GB total | {free_gb:.1f} GB free")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

# 4-bit Qwen3-4B QLoRA (seq-len 4096) + the fp16 merge (~8 GB) wants >= ~20 GB.
if total_gb < 20:
    print()
    print("=" * 64)
    print(f"WARNING: only {total_gb:.0f} GB VRAM -- this looks like a T4.")
    print("This config will likely OOM (especially the fp16 merge in cell 8).")
    print("Fix: Runtime > Change runtime type > L4 (24GB) or A100 (40GB),")
    print("     then Runtime > Restart session and rerun from cell 1.")
    print("=" * 64)
else:
    print(f"OK -- {total_gb:.0f} GB fits 4-bit Qwen3-4B QLoRA (can raise seq-len toward 8192).")

## 2. Get the data
**Do ONE of these** (the notebook will not invent data):
- **Clone your repo:** set `REPO_URL` below to your GitHub repo. The committed `data/sft/{train,dev}.jsonl` splits are used as-is.
- **Upload:** put `train.jsonl`/`dev.jsonl` in a Colab folder and set `DATA_DIR` to it (leave `REPO_URL` as the placeholder).

The raw per-run outputs (`runs/`) are gitignored, so a fresh clone has none; the committed splits are already merged + upsampled, so we use them directly and only rebuild when `runs/` is present -- we never clobber the committed data with empties.

In [ ]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM.git"   # <-- REQUIRED unless you set DATA_DIR to an upload
DATA_DIR = None            # <-- set to an uploaded folder to skip the clone
import os

if DATA_DIR is None:
    assert "<REPO>" not in REPO_URL, (
        "Set REPO_URL to your repo, OR set DATA_DIR to a folder containing "
        "train.jsonl/dev.jsonl that you uploaded via the Files panel."
    )
    if not os.path.exists("slm"):
        !git clone -q $REPO_URL slm
    DATA_DIR = "slm/data/sft"
    # committed splits are already merged+upsampled; only rebuild if the
    # (gitignored) raw per-run outputs are present -- never clobber with empties.
    if os.path.exists("slm/runs"):
        !cd slm && python pipeline/merge_dataset.py --upsample 11=3,15=3

for _f in ("train.jsonl", "dev.jsonl"):
    assert os.path.exists(f"{DATA_DIR}/{_f}"), f"missing {_f} in {DATA_DIR}"
print("data dir:", DATA_DIR, os.listdir(DATA_DIR))

## 3. Config
Hyperparameters. **Batch size + gradient checkpointing are auto-tuned to the GPU** detected in cell 1b: the *effective* batch stays ~16 (the right convergence target for ~2.1k rows) while the *per-device* batch scales with VRAM, and checkpointing is turned off when there's memory to spare (~28% faster). Wall-clock is set by rows×epochs×seq, not by batch size — bigger per-device batch just improves GPU utilization.

In [ ]:
# Qwen3-4B instruct. Confirm the exact HF id (alts: "Qwen/Qwen3-4B",
# "Qwen/Qwen3-4B-Instruct"). Start from Instruct for fast SFT.
model_name  = "Qwen/Qwen3-4B-Instruct-2507"  # base model to fine-tune FROM
adapter_dir = "qwen3-4b-crossword-qlora"      # OUTPUT: trained LoRA adapter dir (merged -> adapter_dir + "-merged")
output_dir  = "results"                       # trainer checkpoints + logs

# QLoRA / LoRA
lora_r, lora_alpha, lora_dropout = 32, 64, 0.05
# Qwen attention + MLP projections
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# programs are long (a full generator); give the sequence room
max_seq_length = 4096

num_train_epochs = 1

# ---- throughput config: auto-tuned to the GPU detected in cell 1b ----
# Wall-clock is set by rows x epochs x seq-len, NOT batch size. We hold the
# EFFECTIVE batch at ~16 (right convergence target for ~2.1k rows) and scale the
# PER-DEVICE batch with VRAM (better GPU utilization; no effect on the optimizer).
# Gradient checkpointing stays ON on every GPU: at seq-len 4096 a 4B model OOMs
# WITHOUT it even on a 40GB A100 (measured) -- batch>=2 activations exceed 40GB, and
# group_by_length packs the longest sequences together, spiking peak memory.
_vram = globals().get("total_gb", 16.0)   # from cell 1b; fallback = conservative 16GB
gradient_checkpointing = True
if _vram >= 38:        # A100 40GB -- headroom for a bigger micro-batch (better GEMM)
    per_device_train_batch_size, gradient_accumulation_steps = 4, 4
elif _vram >= 22:      # L4 24GB
    per_device_train_batch_size, gradient_accumulation_steps = 2, 8
else:                  # T4 16GB (or unknown) -- memory-bound, keep it minimal
    per_device_train_batch_size, gradient_accumulation_steps = 1, 16
per_device_eval_batch_size = per_device_train_batch_size
eff = per_device_train_batch_size * gradient_accumulation_steps
print(f"[auto] VRAM~{_vram:.0f}GB -> per_device_batch={per_device_train_batch_size}, "
      f"accum={gradient_accumulation_steps} (effective ~{eff}), "
      f"grad_checkpointing={gradient_checkpointing}")

learning_rate = 2e-4
lr_scheduler_type = "cosine"
warmup_ratio = 0.03
weight_decay = 0.0
logging_steps = 10

## 4. Load Qwen3-4B in 4-bit (QLoRA) + LoRA adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

# T4 (Colab free tier) has no bf16 -> fall back to fp16 automatically.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
print(f"GPU: {gpu} | bf16 supported: {bf16_ok} -> compute dtype {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map={"": 0},
    torch_dtype=compute_dtype, trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=gradient_checkpointing,   # auto-set in cell 3
    # reentrant=True: Qwen3 saves a different tensor count on recompute, which trips
    # the non-reentrant checkpointer ("A different number of tensors was saved...").
    gradient_checkpointing_kwargs={"use_reentrant": True},
)

peft_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
)

## 5. Load data + render the Qwen chat template
Each row is `{messages:[system,user,assistant]}`. We render it with the model's own chat template into a `text` field; the response-only collator (next cell) then masks everything up to the assistant turn so loss is computed **only on the program**.

In [ ]:
import json
from datasets import Dataset, DatasetDict

# Load ONLY the `messages` column. The per-row `meta` is curation-only and has an
# INCONSISTENT schema across sources (template rows add meta.engine/selection/subset
# + effective_spec.approach; effective_spec.time_budget_s is mixed int/float), so
# load_dataset("json", ...) fails inferring one Arrow struct for meta. messages is
# uniform (list of {role,content} strings), so we drop meta entirely.
def _load_split(path):
    rows = [{"messages": json.loads(l)["messages"]}
            for l in open(path, encoding="utf-8") if l.strip()]
    return Dataset.from_list(rows)

ds = DatasetDict({
    "train": _load_split(f"{DATA_DIR}/train.jsonl"),
    "dev":   _load_split(f"{DATA_DIR}/dev.jsonl"),
})

def render(row):
    # add_generation_prompt=False -> include the assistant turn as the target
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False,
                                                   add_generation_prompt=False)}

ds = ds.map(render, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print("\n--- one rendered example (head) ---\n", ds["train"][0]["text"][:600])

## 6. Response-only loss
Qwen renders the assistant turn after `<|im_start|>assistant\n`. Masking up to that marker means gradients flow only through the generated program, not the (fixed) system contract or user prompt.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "<|im_start|>assistant\n"   # Qwen chat-template assistant marker
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
# sanity: confirm the marker tokenizes and is found in a sample
assert response_template in ds["train"][0]["text"], "assistant marker not found — check template"

## 7. Train (dev = in-training validation; eval stays untouched)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    weight_decay=weight_decay,
    logging_steps=logging_steps,
    optim="paged_adamw_8bit",   # QLoRA-standard 8-bit optimizer: less optimizer memory -> room for a bigger batch
    group_by_length=True,       # bucket similar-length rows so batches are not padded up to 4096 (matters once batch>1)
    bf16=bf16_ok,
    fp16=not bf16_ok,
    gradient_checkpointing=gradient_checkpointing,   # auto-set in cell 3
    gradient_checkpointing_kwargs={"use_reentrant": True},   # must match cell 4 (fixes Qwen3 CheckpointError)
    max_length=max_seq_length,   # canonical arg; max_seq_length is deprecated/ignored
    dataset_text_field="text",
    packing=False,   # required for response-only masking
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["dev"],
    processing_class=tokenizer,   # tokenize the `text` field with our configured tokenizer
    peft_config=peft_config,
    data_collator=collator,
)
trainer.train()

## 8. Save LoRA adapter + merged fp16 model

In [ ]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved LoRA adapter to", adapter_dir)

# merge to a standalone fp16 model for inference / GGUF export
from peft import PeftModel
import torch, gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                            device_map={"": 0}, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.save_pretrained(adapter_dir + "-merged")
tokenizer.save_pretrained(adapter_dir + "-merged")
print("saved merged model to", adapter_dir + "-merged")

## 9. Persist to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_ckpt
# {adapter_dir} is interpolated by IPython from the Python namespace at runtime
!cp -r {adapter_dir} {adapter_dir}-merged /content/drive/MyDrive/slm_ckpt/ 2>/dev/null; echo saved

## Next
Your trained artifacts are in Drive (`MyDrive/slm_ckpt/`): the LoRA adapter (`qwen3-4b-crossword-qlora`) and the merged fp16 model (`…-merged`) for inference / GGUF export.

Eval is run **separately** in `colab_eval_tuned.ipynb` (EVAL 2 on the tuned model). The goal is the base-vs-tuned comparison in `GAP_ANALYSIS.md`: score the tuned model on the pristine held-out `eval.jsonl` through the sandbox+scorer and compare against unaugmented Opus (~5–7% valid) — target is high pass@1.